# 04 — Named Entity Recognition

Find *things* in text: people, places, dates, times, numbers. This notebook
moves through the same arc as the project:

**rules (regex + gazetteers) → spaCy statistical NER → hybrid composition →
span-level evaluation**

Production use: `backend/app/nlp/entities.py` · Docs: `docs/nlp/04_entity_extraction.md`

In [ ]:
import re, sys
from pathlib import Path

cwd = Path.cwd()
backend_dir = cwd.parent / "backend" if cwd.name == "notebooks" else cwd / "backend"
sys.path.insert(0, str(backend_dir))

from app.nlp.entities import (
    ENTITY_LABELS, RuleBasedEntityExtractor, SpacyEntityExtractor,
    get_entity_extractor,
)
print("labels:", ENTITY_LABELS)
print("active extractor:", get_entity_extractor().name)

## 1. BIO tagging — how NER is modeled

Sequence labeling tags **every token**: `B-` (begin), `I-` (inside), `O`
(outside). Multi-token entities are why this exists: "New York" needs
B-LOCATION I-LOCATION — a per-word classifier can't know where one entity
ends and the next begins.

Our API exposes the equivalent as **character offsets** (start/end into the
original message) so the frontend can highlight spans in place.

In [ ]:
def to_bio(text, spans):
    tags = ["O"] * len(text.split())
    # toy demonstration on words (production uses char offsets)
    words = text.split()
    cursor = 0
    positions = []
    for w in words:
        start = text.index(w, cursor); positions.append((start, start + len(w)))
        cursor = start + len(w)
    for span in spans:
        for i, (s, e) in enumerate(positions):
            if s >= span["start"] and e <= span["end"]:
                first = s == span["start"]
                tags[i] = f"{'B' if first else 'I'}-{span['label']}"
    return list(zip(text.split(), tags))

demo = "Dr. Smith flies to Paris on 2026-10-08"
spans = [{"start": 0, "end": 10, "label": "PERSON"},
         {"start": 19, "end": 24, "label": "LOCATION"},
         {"start": 28, "end": 38, "label": "DATE"}]
for word, tag in to_bio(demo, spans):
    print(f"{word:12} {tag}")

## 2. Rule-based extraction

Regexes for regex-shaped phenomena (TIME, DATE, NUMBER) + gazetteers for
known names (PERSON, LOCATION). Priority order resolves overlaps: TIME
claims `3:30pm` before NUMBER can slice it.

In [ ]:
rules = RuleBasedEntityExtractor()
samples = [
    "Dr. Smith meets Ada in Paris at 3:30pm tomorrow with 42 files",
    "I paid 1,250.50 dollars on 2026-10-08",
    "the 3:30 train on 2026-10-08 cost 25",
    "the quick brown fox jumps",
]
for s in samples:
    print(repr(s))
    for e in rules.extract(s):
        ok = s[e.start:e.end] == e.text
        print(f"   {e.label:9} {e.text!r:20} [{e.start}:{e.end}] offsets_ok={ok}")
    print()

## 3. Where rules break

1. Unknown names: `Zaphod Beeblebrox` without a title or list entry → missed.
2. Ambiguity: "Apple" — fruit or company? Rules have no context at all.
3. Every gazetteer is incomplete forever.

This is what statistical models are for.

In [ ]:
for s in ["Zaphod Beeblebrox visited Berlin", "I love apples from Washington"]:
    got = [e.text for e in rules.extract(s)]
    print(f"{s!r:50} rules found: {got}")

## 4. Statistical NER with spaCy

`en_core_web_sm` labels from learned context. Its ontology (PERSON, GPE,
CARDINAL, MONEY...) is mapped onto ours — anything unmapped (ORG, PRODUCT)
is dropped rather than force-fit.

In [ ]:
spacy_ex = SpacyEntityExtractor()
for s in samples:
    print(repr(s))
    for e in spacy_ex.extract(s):
        print(f"   {e.label:9} {e.text!r:24} conf={e.confidence} src={e.source}")
    print()

## 5. The hybrid — compose, don't choose

spaCy alone mislabels `3:30pm` as CARDINAL in short sentences; rules alone
miss `Ada`. The hybrid layering:

1. rule TIME/DATE claim spans (precise patterns)
2. spaCy adds context-derived spans (no overlap)
3. remaining rules fill gaps (gazetteers, titled names)

In [ ]:
hybrid = get_entity_extractor()
test = "the 3:30pm slot for Dr. Smith and Ada in Paris"
for e in hybrid.extract(test):
    print(f"   {e.label:9} {e.text!r:20} [{e.start}:{e.end}] src={e.source}")

## 6. Span-level evaluation

Exact match: same text, same offsets, same label = 1 correct. Then the usual
precision/recall/F1 — but counting **spans**, not tokens.

In [ ]:
def score(pred, gold):
    pred = {(e["text"], e["label"]) for e in pred}
    gold = {(e["text"], e["label"]) for e in gold}
    tp = len(pred & gold)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold) if gold else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f1

gold = [
    {"text": "Smith", "label": "PERSON"}, {"text": "Ada", "label": "PERSON"},
    {"text": "Paris", "label": "LOCATION"}, {"text": "42", "label": "NUMBER"},
]
text = "Smith meets Ada in Paris with 42 files"
for name, ex in [("rules", rules), ("spacy", spacy_ex), ("hybrid", hybrid)]:
    pred = [{"text": e.text, "label": e.label} for e in ex.extract(text)]
    p, r, f1 = score(pred, gold)
    print(f"{name:6} precision={p:.2f} recall={r:.2f} F1={f1:.2f}  pred={pred}")

## 7. Production behavior

`ChatService` runs the active extractor after intent classification and
returns spans in `ChatResponse.entities`. If the spaCy model is missing the
factory silently falls back to rules — the endpoint never fails because of a
model file.

In [ ]:
from app.schemas import ChatRequest
from app.services.chat_service import ChatService

svc = ChatService()
resp = svc.handle(ChatRequest(message="meet me in Paris at noon tomorrow"))
print("intent:", resp.intent, f"({resp.confidence})")
for e in resp.entities:
    print(f"  {e.label:9} {e.text!r:12} [{e.start}:{e.end}] conf={e.confidence}")